In [1]:
# required imports
from pathlib import Path
import pandas as pd

In [2]:
project_dir = Path.cwd().parent.parent
interim_dir = project_dir / "interim"

data = {
    year: pd.read_csv(interim_dir / f"opencity_electiondata_{year}.csv")
    for year in [2014, 2019, 2024]
}

In [9]:
# check missing values in each dataset

for year, df in data.items():
    print(f"\n{year}")
    print(df.isna().sum())


2014
Year                     0
State_Name               0
Constituency_No          0
Constituency_Name        0
Candidate                0
Party                    0
Position                 0
Votes                    0
Vote_Share_Percentage    0
Margin                   0
dtype: int64

2019
Year                     0
State_Name               0
Constituency_No          0
Constituency_Name        0
Candidate                0
Party                    2
Position                 0
Votes                    0
Vote_Share_Percentage    0
Margin                   0
dtype: int64

2024
State           0
PC No           0
PC Name         0
Candidate       0
Party           0
EVM Votes       0
Postal Votes    0
Total Votes     0
dtype: int64


### Missing-value inspection

- **2014:** No missing values detected.
- **2019:** Two missing values in `Party`; all other columns are complete.
- **2024:** No missing values detected by pandas.
- **Note:** This check detects standard missing values, but not placeholders such as `-`, empty strings, or text like `NA`.


In [17]:
df = data[2019]
print(df[df["Party"].isna()])

      Year   State_Name  Constituency_No Constituency_Name Candidate Party  \
8599  2019        Bihar               23        SAMASTIPUR      NOTA   NaN   
8609  2019  Maharashtra               45            SATARA      NOTA   NaN   

      Position    Votes  Vote_Share_Percentage   Margin  
8599         4  25694.0                   3.26  10797.0  
8609         5  10159.0                   0.81   5484.0  


### Missing party values (2019)

Two rows have missing `Party` values, both corresponding to `NOTA` candidates in Bihar and Maharashtra. Although the party names are missing, both rows contain position, vote count, vote share, and margin values. These appear to be records for NOTA rather than ordinary candidates, so the missing party values may be expected. This should be verified against the original source before any imputation or removal.

In [12]:
# find blank strings and common missing-value placeholders

markers = ["", "-", "--", "NA", "N/A", "NULL", "null", "None", "nan"]

for year, df in data.items():
    print(f"\n{'=' * 8} {year} {'=' * 8}")

    for column in df.select_dtypes(include="object").columns:
        values = df[column].astype("string").str.strip()
        matches = values[values.isin(markers)]

        if not matches.empty:
            print(f"{column}: {matches.value_counts().to_dict()}")


======== 2014 ========

======== 2019 ========

======== 2024 ========
EVM Votes: {'-': 1}
Postal Votes: {'-': 578}
Total Votes: {'-': 1}


### Hidden missing-value markers

- **2014 and 2019:** No checked placeholders were found.
- **2024:** `-` appears once in `EVM Votes`, 578 times in `Postal Votes`, and once in `Total Votes`.
- The meaning of `-` needs to be checked before deciding how to handle these values.

In [13]:
# check numeric ranges and vote consistency

for year, df in data.items():
    print(f"\n{'=' * 8} {year} {'=' * 8}")

    if year in [2014, 2019]:
        for column in ["Position", "Votes", "Vote_Share_Percentage", "Margin"]:
            values = pd.to_numeric(df[column], errors="coerce")
            print(f"{column}: negative={(values < 0).sum()}, missing={values.isna().sum()}")

        share = pd.to_numeric(df["Vote_Share_Percentage"], errors="coerce")
        print("vote shares outside 0–100:", ((share < 0) | (share > 100)).sum())

    else:
        votes = {}
        for column in ["EVM Votes", "Postal Votes", "Total Votes"]:
            votes[column] = pd.to_numeric(df[column], errors="coerce")
            print(f"{column}: negative={(votes[column] < 0).sum()}, non-numeric={votes[column].isna().sum()}")

        complete = (
            votes["EVM Votes"].notna()
            & votes["Postal Votes"].notna()
            & votes["Total Votes"].notna()
        )

        mismatch = (
            votes["EVM Votes"] + votes["Postal Votes"]
            != votes["Total Votes"]
        ) & complete

        print("rows where EVM + postal != total:", mismatch.sum())


======== 2014 ========
Position: negative=0, missing=0
Votes: negative=0, missing=0
Vote_Share_Percentage: negative=0, missing=0
Margin: negative=0, missing=0
vote shares outside 0–100: 0

======== 2019 ========
Position: negative=0, missing=0
Votes: negative=0, missing=0
Vote_Share_Percentage: negative=0, missing=0
Margin: negative=0, missing=0
vote shares outside 0–100: 0

======== 2024 ========
EVM Votes: negative=0, non-numeric=1
Postal Votes: negative=0, non-numeric=578
Total Votes: negative=0, non-numeric=1
rows where EVM + postal != total: 0


### Numeric anomaly inspection

- **2014 and 2019:** No negative values, missing numeric values, or vote shares outside 0–100 were detected in the checked columns.
- **2024:** No negative vote counts were detected. Non-numeric markers (`-`) occurred once in `EVM Votes`, 578 times in `Postal Votes`, and once in `Total Votes`.
- **Vote consistency:** All rows with numeric values in all three vote columns satisfied `EVM Votes + Postal Votes = Total Votes`.
- The meaning of `-` should be established before replacing it with zero.

In [18]:
df = data[2024]

postal = pd.to_numeric(df["Postal Votes"].replace("-", "0"), errors="coerce")
evm = pd.to_numeric(df["EVM Votes"], errors="coerce")
total = pd.to_numeric(df["Total Votes"], errors="coerce")

complete = evm.notna() & postal.notna() & total.notna()

print("Dash entries:", (df["Postal Votes"].astype(str).str.strip() == "-").sum())
print("Postal vote sum:", int(postal.sum()))
print("Postal vote sum (lakh):", round(postal.sum() / 100000, 2))
print("Vote total mismatches:", ((evm + postal != total) & complete).sum())

Dash entries: 578
Postal vote sum: 3744508
Postal vote sum (lakh): 37.45
Vote total mismatches: 0


### Postal votes

The 2024 dataset contains 578 `-` entries in `Postal Votes`. Treating these as zero gives 37.45 lakh postal votes, with no mismatches between `EVM Votes + Postal Votes` and `Total Votes`.

The ECI reported 42.82 lakh postal ballots, of which approximately 5.35 lakh were rejected, giving around 37.47 lakh valid ballots. This is close to the dataset total. ([ECI](https://www.pib.gov.in/PressReleasePage.aspx?PRID=2088064), [Deccan Herald](https://www.deccanherald.com/india/unprecedented-compared-to-other-democracies-ec-shares-data-on-lok-sabha-polls-3332443))

The checks support treating `-` as zero, although the OpenCity dataset does not explicitly define it.

In [19]:
for year, df in data.items():
    print(f"\n{'=' * 8} {year} {'=' * 8}")
    print(df.select_dtypes(include="number").describe().T)


======== 2014 ========
                        count          mean            std      min      25%  \
Year                   8843.0   2014.000000       0.000000  2014.00  2014.00   
Constituency_No        8843.0     19.523917      17.040970     1.00     7.00   
Position               8843.0      9.767613       6.730518     1.00     5.00   
Votes                  8843.0  63173.261450  141580.649917   105.00  1766.50   
Vote_Share_Percentage  8843.0      6.196986      13.594145     0.01     0.17   
Margin                 8843.0  29756.836142   78627.801974     0.00   100.00   

                           50%       75%        max  
Year                   2014.00   2014.00    2014.00  
Constituency_No          15.00     28.00      80.00  
Position                  9.00     13.00      43.00  
Votes                  4520.00  15321.50  922416.00  
Vote_Share_Percentage     0.44      1.57      75.79  
Margin                  530.00   4398.50  696321.00  

======== 2019 ========
             